# Study 920 — Total Cost of Ownership — the teardown

The chained-period tracking-difference estimator, why the bootstrap runs on years rather than months, the adjustment-artefact autopsy, the break-even curve and its interval, the overlapping holding-period race with one execution lag, the borrow sweep that kills the long/short version, and the live calibration check.

Every real-tape number is frozen from `docs/results.md` — common window `2020-10-13 → 2026-06-30`, 1,434 days, fingerprint `c3aa747fad51`, as-of 2026-06-30. Total-return closes (`auto_adjust=True`); tracking difference is always **cheap minus liquid, bp/yr**.

In [1]:
R = {'asof': '2026-06-30', 'fp_all': 'c10bd02337b7', 'fp_common': 'c3aa747fad51', 'cw_start': '2020-10-13', 'cw_end': '2026-06-30', 'cw_days': 1434, 'cw_years': 5, 'er_spy': 9.45, 'er_ivv': 3.0, 'er_voo': 3.0, 'er_qqq': 20.0, 'er_qqqm': 15.0, 'ivv_gap': 6.45, 'ivv_td': 5.78, 'ivv_sd': 4.43, 'ivv_t': 2.92, 'ivv_lo': 3.55, 'ivv_hi': 7.9, 'ivv_tlo': 0.2806, 'ivv_thi': 11.2827, 'ivv_pos': 5, 'ivv_exact': 5.7816, 'voo_gap': 6.45, 'voo_td': 6.6, 'voo_sd': 4.34, 'voo_t': 3.4, 'voo_lo': 3.46, 'voo_hi': 9.33, 'voo_tlo': 1.2141, 'voo_thi': 11.9931, 'voo_pos': 5, 'voo_exact': 6.6036, 'pla_gap': 0.0, 'pla_td': 0.82, 'pla_sd': 2.46, 'pla_t': 0.75, 'pla_lo': -0.25, 'pla_hi': 2.21, 'pla_tlo': -2.2369, 'pla_thi': 3.8809, 'pla_pos': 3, 'pla_exact': 0.822, 'qqm_gap': 5.0, 'qqm_td': 7.19, 'qqm_sd': 3.05, 'qqm_t': 5.27, 'qqm_lo': 4.78, 'qqm_hi': 9.48, 'qqm_tlo': 3.4003, 'qqm_thi': 10.9794, 'qqm_pos': 5, 'qqm_exact': 7.1898, 'qqm_years': (2.2, 8.8, 10.1, 8.4, 6.5), 'ivv_cum': 3.38, 'ivv_mon': 3.9, 'ivv_head': 5.7, 'ivv_yrs': 28.9, 'ivv_tail': -15.4, 'voo_cum': 8.07, 'voo_mon': 9.04, 'voo_head': 4.7, 'voo_yrs': 33.0, 'voo_tail': 8.2, 'pla_cum': 4.69, 'pla_mon': 5.14, 'pla_head': -1.0, 'pla_yrs': 4.1, 'pla_tail': 23.6, 'qqm_cum': 4.48, 'qqm_mon': 2.23, 'qqm_head': -5.0, 'qqm_yrs': 35.9, 'qqm_tail': -5.5, 'full_ivv_td': 2.24, 'full_ivv_t': 0.78, 'full_ivv_trim': 3.29, 'full_ivv_trim_t': 1.41, 'full_ivv_worst_y': 2016, 'full_ivv_worst': -40.9, 'full_ivv_n': 25, 'full_voo_td': 2.6, 'full_voo_t': 0.69, 'full_voo_trim': 5.48, 'full_voo_trim_t': 4.65, 'full_voo_worst_y': 2014, 'full_voo_worst': -47.3, 'full_voo_n': 15, 'full_pla_td': 0.86, 'full_pla_t': 0.15, 'full_pla_worst_y': 2014, 'full_pla_worst': -56.5, 'need_ivv': 165, 'need_pla': 2534, 'era_ivv_early': 1.14, 'era_ivv_early_t': 0.3, 'era_ivv_late': 5.78, 'era_ivv_late_t': 2.92, 'era_voo_early': 0.73, 'era_voo_early_t': 0.12, 'era_voo_late': 6.6, 'era_voo_late_t': 3.4, 'era_qqm_early': 5.52, 'era_qqm_early_t': 1.68, 'era_qqm_late': 7.42, 'era_qqm_late_t': 7.93, 'be_ivv': (22, 44, 87, 218), 'be_voo': (19, 38, 76, 191), 'be_qqm': (18, 35, 70, 175), 'be_pla': (153, 307, 613, 1533), 'be_ci_ivv': 71, 'be_ci_voo': 73, 'be_ci_qqm': 53, 'be_t_ivv': 898, 'be_t_voo': 208, 'be_t_qqm': 74, 'emp_ivv': {21: (-0.46, 0.432, -3.25), 63: (0.72, 0.62, 2.34), 126: (2.64, 0.777, 3.84), 252: (6.76, 0.855, 4.07), 504: (16.65, 0.992, 4.31), 756: (30.33, 0.998, 7.33), 1008: (39.6, 1.0, 35.83)}, 'emp_voo': {21: (-0.4, 0.433, -3.23), 63: (0.82, 0.631, 2.76), 126: (2.73, 0.788, 4.15), 252: (6.86, 0.869, 4.12), 504: (16.94, 0.995, 4.35), 756: (30.66, 0.999, 7.7), 1008: (40.78, 1.0, 36.41)}, 'emp_qqm': {21: (-0.51, 0.436, -3.05), 63: (0.73, 0.613, 2.68), 126: (2.65, 0.79, 5.19), 252: (7.47, 0.921, 6.11), 504: (19.61, 0.987, 6.65), 756: (36.81, 0.994, 10.43), 1008: (48.91, 0.993, 57.3)}, 'emp_pla': {21: (-0.94, 0.33, -10.54), 63: (-0.91, 0.35, -6.32), 126: (-0.91, 0.368, -4.37), 252: (-0.9, 0.373, -3.55), 504: (-0.71, 0.42, -4.12), 756: (-0.67, 0.402, -3.48), 1008: (0.18, 0.511, 0.35)}, 'emp_first_pos': 42, 'emp_first_sig': 63, 'emp_first_pos_3bp': 126, 'emp_first_sig_3bp': 189, 'ls_ivv_gross': 0.95, 'ls_voo_gross': 4.53, 'ls_qqm_gross': 1.93, 'ls_pla_gross': 3.58, 'ls_noise_ivv': 71, 'ls_noise_voo': 77, 'ls_noise_qqm': 90, 'ls_noise_pla': 52, 'ls_qqm_net5': -3.77, 'ls_qqm_net10': -8.77, 'ls_qqm_net25': -23.77, 'sh_spy': 0.7802, 'sh_ivv': 0.7878, 'sh_voo': 0.7932, 'sh_qqq': 0.7233, 'sh_qqqm': 0.7289, 'sh_t_ivv': 0.13, 'sh_t_voo': 0.59, 'sh_t_qqm': 0.11, 'sh_diff_pla': 0.0054, 'sh_t_pla': 0.59, 'arm_vol': 16.8, 'arm_vol_q': 22.5, 'daily_sd_bp': 5.0, 'mon_sd_qqm': 10.0, 'ann_sd_qqm': 3.0, 'syn': ((0.0, -0.04, -0.02), (3.0, 3.41, 1.19), (6.0, 6.31, 3.75), (12.0, 13.63, 6.62)), 'syn_null_mean_t': -0.04, 'syn_null_sd_t': 0.34, 'syn_null_fires': 0, 'syn_null_n': 8}

## 1. The estimand and the estimator

Two wrappers on one index. Model the log price ratio as a drift plus stationary noise:

$$\log\frac{P^{cheap}_t}{P^{liquid}_t} = \alpha t + \varepsilon_t$$

where $\alpha$ is the annual tracking-difference advantage — fee gap, lending revenue, sampling error, cash drag, all of it — and $\varepsilon_t$ is closing-print noise plus a slow level wobble. We report three estimators of $\alpha$: the **cumulative** drift, the mean of **chained complete calendar years**, and the mean of **chained complete months** with a Newey-West *t*.

Chaining matters. Tracking difference does not arrive smoothly — it lands in steps on distribution dates, which cluster at quarter ends. Measuring first-to-last *within* each period silently drops every period-boundary gap, i.e. exactly the sessions carrying the signal, and the three estimators then disagree with each other. Chaining removes *that* source of disagreement — but not the one that matters here, which is §2b.

> 💡 **In plain words:** measure the gap from one year-end to the next, not from January to December — otherwise you throw away the New Year's gap, and that is where the money moves.

## 2. Why the bootstrap runs on years, not months

$\varepsilon_t$ is a **level** error, so chained differences of it are strongly *negatively* autocorrelated and telescope. On QQQ/QQQM the monthly tracking differences have a standard deviation near 10 bp while their twelve-month sums — which are just twelve of them added up — have one near 3 bp. Under independence the annual figure would be $\sqrt{12} \times 10 \approx 35$ bp. A monthly-frequency interval therefore overstates the uncertainty by an order of magnitude, and a Bartlett kernel does not rescue it (the truncation throws away most of the negative mass). So the circular block bootstrap runs on complete years with two-year blocks.

**And running it on years costs sample size: the common window has five.** A percentile block bootstrap on five points resamples an empirical distribution built from the very five numbers whose dispersion it is trying to price, with no allowance for $\hat{\sigma}$ being estimated from them. It comes out far too narrow. Every table below therefore carries a **Student-*t* interval beside the bootstrap one**, and every pessimistic claim in this study quotes the *t* interval.

## 2b. The three estimators disagree — the study's sharpest caveat

Chaining makes the estimators consistent *over the periods they share*. It cannot reconcile them when they cover **different** spans, and they do: the annual estimator keeps complete calendar years only, the cumulative one keeps the partial stubs at both ends too. On the real common window that difference is not cosmetic:

| Pair | **Annual** | Cumulative | Monthly ×12 | opening stub | complete years | closing stub |
|---|--:|--:|--:|--:|--:|--:|
| IVV over SPY | **+5.78** | +3.38 | +3.90 | +5.7 | +28.9 | **-15.4** |
| VOO over SPY | **+6.60** | +8.07 | +9.04 | +4.7 | +33.0 | +8.2 |
| VOO over IVV *(placebo)* | **+0.82** | **+4.69** | +5.14 | -1.0 | +4.1 | **+23.6** |
| QQQM over QQQ | **+7.19** | +4.48 | +2.23 | -5.0 | +35.9 | -5.5 |

The stubs are 2020 Q4 and 2026 H1, in bp of total drift. **Two funds charging the same 3 bp cannot really drift +23.6 bp apart in six months** — that is a level artefact in one closing print, four times the whole annual signal, and it is what lifts the placebo's *cumulative* estimate to +4.69 bp/yr, **above IVV-over-SPY's +3.38**.

So the headline is estimator-conditional, and honesty requires saying so plainly: a reader who prefers the raw cumulative drift gets **no signal at all** on the IVV/SPY pair. The defence is not that the annual estimator is larger. It is that the complete-calendar-period rule was fixed ex ante by the as-of convention, and that the placebo — whose true gap is zero by construction — demonstrates exactly the contamination the rule removes. Two of three estimators (annual, and the overlapping race in §7) leave the placebo quiet. The cumulative one does not.

## 3. The headline — common window, fixed by QQQM's inception

The window is the only span on which all six wrappers trade. It was chosen by an inception date, not by a result — and the placebo pair is the guard against the suspicion that it was. Note that **all four pairs rest on the same five complete calendar years (2021–2025)**, including the S&P pairs whose own histories run to 15 and 25 years: *n* = 5 everywhere, so these are not four independent samples.

The sign count is the one statement that survives without any interval at all: 5/5 positive years is a one-sided sign-test *p* of 1/32 = 0.031.

In [2]:
hdr = ('pair', 'stated', 'realised', 'sd', 't', 'yrs+', 'boot lo', 'boot hi', 't-CI lo', 't-CI hi')
print('%-24s %7s %9s %5s %6s %5s %8s %8s %8s %8s' % hdr)
rows = [('IVV over SPY', 'ivv'), ('VOO over SPY', 'voo'),
        ('QQQM over QQQ', 'qqm'), ('VOO over IVV [placebo]', 'pla')]
for name, k in rows:
    print('%-24s %+7.2f %+9.2f %5.2f %+6.2f %4d/5 %+8.2f %+8.2f %+8.2f %+8.2f'
          % (name, R[k+'_gap'], R[k+'_td'], R[k+'_sd'], R[k+'_t'], R[k+'_pos'],
             R[k+'_lo'], R[k+'_hi'], R[k+'_tlo'], R[k+'_thi']))
print('\nthe bootstrap interval is ~2-3x too narrow at n=5; the t interval is the honest one.')
print('all three fee-gap pairs still clear zero on it -- IVV over SPY only just (%+.2f).'
      % R['ivv_tlo'])
print('\nQQQM annual tracking differences (bp): %s  -> positive in 5/5 years'
      % ', '.join('%+.1f' % v for v in R['qqm_years']))

pair                      stated  realised    sd      t  yrs+  boot lo  boot hi  t-CI lo  t-CI hi
IVV over SPY               +6.45     +5.78  4.43  +2.92    5/5    +3.55    +7.90    +0.28   +11.28
VOO over SPY               +6.45     +6.60  4.34  +3.40    5/5    +3.46    +9.33    +1.21   +11.99
QQQM over QQQ              +5.00     +7.19  3.05  +5.27    5/5    +4.78    +9.48    +3.40   +10.98
VOO over IVV [placebo]     +0.00     +0.82  2.46  +0.75    3/5    -0.25    +2.21    -2.24    +3.88

the bootstrap interval is ~2-3x too narrow at n=5; the t interval is the honest one.
all three fee-gap pairs still clear zero on it -- IVV over SPY only just (+0.28).

QQQM annual tracking differences (bp): +2.2, +8.8, +10.1, +8.4, +6.5  -> positive in 5/5 years


## 4. The autopsy — why the full histories say less

Run the same estimator over each pair's whole life and the S&P pairs collapse:

| Pair | Window | TD | *t* | Trimmed (*t*) | Worst single year |
|---|---|--:|--:|--:|--:|
| IVV over SPY | 25 yrs | +2.24 | +0.78 | +3.29 (+1.41) | 2016: **-40.9 bp** |
| VOO over SPY | 15 yrs | +2.60 | +0.69 | +5.48 (**+4.65**) | 2014: **-47.3 bp** |
| VOO over IVV | 15 yrs | +0.86 | +0.15 | — | 2014: **-56.5 bp** |

The diagnostic is the placebo row. Two funds charging the *same* three basis points cannot truly differ by -56 bp in a year. Those are adjustment artefacts in the public total-return series — a distribution timed into the wrong session — and they are an order of magnitude larger than the effect being measured. Drop the best and worst year and VOO-over-SPY lands on +5.48 bp/yr at *t* = +4.65, i.e. on its prospectus gap.

> 💡 **In plain words:** the fee gap did not appear in 2020. Our ability to *see* it did.

## 5. Power, and the era cut

The cleanest way to state the same thing is a power calculation: at the observed annual dispersion, reaching a one-sample |*t*| of 2 would take **165 years** on the full IVV/SPY history and **2,534 years** on the placebo pair. On the common window the dispersion collapses to 3–4 bp and five years suffice.

In [3]:
print('era cut, full histories, split 2020-01-01 (bp/yr, t)')
for name, a, at, b, bt in [
        ('IVV over SPY', R['era_ivv_early'], R['era_ivv_early_t'], R['era_ivv_late'], R['era_ivv_late_t']),
        ('VOO over SPY', R['era_voo_early'], R['era_voo_early_t'], R['era_voo_late'], R['era_voo_late_t'])]:
    print('  %-14s early %+5.2f (t %+.2f)   late %+5.2f (t %+.2f)' % (name, a, at, b, bt))
print('  %-14s early %+5.2f (t %+.2f)   late %+5.2f (t %+.2f)   <- split 2023-01-01'
      % ('QQQM over QQQ', R['era_qqm_early'], R['era_qqm_early_t'],
         R['era_qqm_late'], R['era_qqm_late_t']))
print('\nyears of tape needed for |t|=2 at the observed dispersion:')
print('  full IVV/SPY history: %d   placebo pair: %d' % (R['need_ivv'], R['need_pla']))

era cut, full histories, split 2020-01-01 (bp/yr, t)
  IVV over SPY   early +1.14 (t +0.30)   late +5.78 (t +2.92)
  VOO over SPY   early +0.73 (t +0.12)   late +6.60 (t +3.40)
  QQQM over QQQ  early +5.52 (t +1.68)   late +7.42 (t +7.93)   <- split 2023-01-01

years of tape needed for |t|=2 at the observed dispersion:
  full IVV/SPY history: 165   placebo pair: 2534


## 6. The break-even curve — the spread is a swept ASSUMPTION

$H^\ast = \Delta\text{spread} / \alpha \times 252$ trading days. No daily-close tape carries quotes, so $\Delta\text{spread}$ is swept end to end rather than assumed. The cell below runs the arithmetic live on the frozen real-tape $\alpha$s (unrounded, so it reproduces `docs/results.md` to the day), showing the point estimate and **both** intervals' pessimistic ends so the gap between them is visible.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from tco import strategy as st

def _d(a, s):
    v = st.breakeven_days(a, s)
    return '%8s' % ('never' if v == float('inf') else '%.0f' % v)

grid = (0.5, 1.0, 2.0, 3.0, 5.0, 10.0)
print('%-24s %s' % ('break-even (trading days)', '  '.join('%6.1fbp' % s for s in grid)))
for name, k in [('IVV over SPY', 'ivv'), ('VOO over SPY', 'voo'),
                ('QQQM over QQQ', 'qqm'), ('VOO over IVV [placebo]', 'pla')]:
    print('%-24s %s   <- point' % (name, ''.join(_d(R[k+'_exact'], s) for s in grid)))
    print('%-24s %s   <- bootstrap low end (too optimistic)'
          % ('', ''.join(_d(R[k+'_lo'], s) for s in grid)))
    print('%-24s %s   <- Student-t low end (QUOTED)'
          % ('', ''.join(_d(R[k+'_tlo'], s) for s in grid)))

break-even (trading days)    0.5bp     1.0bp     2.0bp     3.0bp     5.0bp    10.0bp
IVV over SPY                   22      44      87     131     218     436   <- point
                               35      71     142     213     355     710   <- bootstrap low end (too optimistic)
                              449     898    1796    2694    4490    8981   <- Student-t low end (QUOTED)
VOO over SPY                   19      38      76     114     191     382   <- point
                               36      73     146     218     364     728   <- bootstrap low end (too optimistic)
                              104     208     415     623    1038    2076   <- Student-t low end (QUOTED)
QQQM over QQQ                  18      35      70     105     175     350   <- point
                               26      53     105     158     264     527   <- bootstrap low end (too optimistic)
                               37      74     148     222     371     741   <- Student-t low end (QUOTED)


At 1 bp the point estimates say 35–44 days. The bootstrap's pessimistic end says 53–73; the Student-*t* end says **74 / 208 / 898 days**. That last row is the one the study quotes, and it is the difference between *two and a half months* and *three and a half years* for IVV/SPY. The direction of the effect is well established; its magnitude is pinned down only to within a factor of about three.

## 7. The empirical race — one execution lag, overlapping windows

The choice is made with information through the close of day *t*; both candidate positions are entered at the close of day *t+1*. The cheap wrapper is charged the extra round trip **once**, at entry — that is the entire execution-cost difference between the two choices, since one round trip is paid either way. Newey-West lags are set to the holding period (the overlap length). One basis point of extra round trip:

| Hold | QQQM over QQQ | VOO over SPY | IVV over SPY | VOO over IVV *(placebo)* |
|---|--:|--:|--:|--:|
| 21 d | -0.51 (44%, -3.05) | -0.40 (43%, -3.23) | -0.46 (43%, -3.25) | *-0.94 (33%, -10.54)* |
| 63 d | +0.73 (61%, +2.68) | +0.82 (63%, +2.76) | +0.72 (62%, +2.34) | *-0.91 (35%, -6.32)* |
| 126 d | +2.65 (79%, +5.19) | +2.73 (79%, +4.15) | +2.64 (78%, +3.84) | *-0.91 (37%, -4.37)* |
| 252 d | +7.47 (92%, +6.11) | +6.86 (87%, +4.12) | +6.76 (86%, +4.07) | *-0.90 (37%, -3.55)* |
| 504 d | +19.61 (99%, +6.65) | +16.94 (100%, +4.35) | +16.65 (99%, +4.31) | *-0.71 (42%, -4.12)* |
| 756 d | +36.81 (99%, +10.43) | +30.66 (100%, +7.70) | +30.33 (100%, +7.33) | *-0.67 (40%, -3.48)* |
| 1008 d | +48.91 (99%, +57.30) | +40.78 (100%, +36.41) | +39.60 (100%, +35.83) | *+0.18 (51%, +0.35)* |

Positive from **42 days**, |*t*| ≥ 2 from **63 days**, on all three real pairs — against an analytic break-even of 35–44 days. At three basis points of extra round trip those become 126 and 189 days. This is the second estimator that leaves the placebo quiet, and the only one that uses every window rather than year-end prints alone: the placebo is negative out to **three** years (-0.67 bp) and only marginally positive at four (+0.18 bp).

> ⚠️ **Read only the short rows as inference.** Newey-West lags are set to the holding period, so the bandwidth-to-sample ratio is 0.05 at 63 days over ~1,370 windows — fine — but 2.4 at 1,008 days over 425 windows, which contain barely one independent draw. The *t* of +57.3 in the bottom row is a bandwidth artefact, not evidence, and the win rates have the same defect because overlapping windows are not independent trials. **The 63-day row is the significance claim; everything past 252 days is descriptive.**

## 8. Why the long/short version is dead

Long the cheap wrapper, short the liquid one, daily-rebalanced, 1 bp one-way on all four legs. This is the study's only short leg, so it is the only place borrow is paid — an ASSUMPTION with no tape behind it, hence swept.

In [5]:
print('common window, long cheap / short liquid, 1 bp one-way x 4 legs')
for name, g, v, tag in [('IVV / SPY', R['ls_ivv_gross'], R['ls_noise_ivv'], ''),
                        ('VOO / SPY', R['ls_voo_gross'], R['ls_noise_voo'], ''),
                        ('QQQM / QQQ', R['ls_qqm_gross'], R['ls_noise_qqm'], ''),
                        ('VOO / IVV', R['ls_pla_gross'], R['ls_noise_pla'],
                         '   <- PLACEBO: no fee gap to harvest')]:
    print('  %-11s gross %+5.2f bp/yr   tracking noise %3d bp/yr   Sharpe ~ %.2f%s'
          % (name, g, v, g / v, tag))
print('\nQQQM / QQQ net of borrow:  5 bp %+.2f   10 bp %+.2f   25 bp %+.2f  bp/yr'
      % (R['ls_qqm_net5'], R['ls_qqm_net10'], R['ls_qqm_net25']))

common window, long cheap / short liquid, 1 bp one-way x 4 legs
  IVV / SPY   gross +0.95 bp/yr   tracking noise  71 bp/yr   Sharpe ~ 0.01
  VOO / SPY   gross +4.53 bp/yr   tracking noise  77 bp/yr   Sharpe ~ 0.06
  QQQM / QQQ  gross +1.93 bp/yr   tracking noise  90 bp/yr   Sharpe ~ 0.02
  VOO / IVV   gross +3.58 bp/yr   tracking noise  52 bp/yr   Sharpe ~ 0.07   <- PLACEBO: no fee gap to harvest

QQQM / QQQ net of borrow:  5 bp -3.77   10 bp -8.77   25 bp -23.77  bp/yr


**The placebo row is the verdict on this construction.** A pair with *no* fee gap harvests +3.58 bp/yr gross — more than IVV/SPY (+0.95) and more than QQQM/QQQ (+1.93). Daily rebalancing of a mean-reverting print spread does not capture a fee; it captures noise. It also gives most of the buy-and-hold gap back (+1.93 bp/yr traded against +7.19 held), and the harvest sits inside 90 bp/yr of tracking noise, so **any borrow above about five basis points a year buries it**. The gap is a selection decision, never a position.

> 💡 **In plain words:** you can keep this money by owning the right fund. You cannot win it by betting on the difference.

## 9. The Sharpe race, and why it is the wrong instrument

Excess-of-cash (BIL) annualised Sharpe on the common window: SPY +0.7802 vs IVV +0.7878 vs VOO +0.7932; QQQ +0.7233 vs QQQM +0.7289 — both legs excess of the **same** cash series, so the cash leg cancels in every difference. HAC *t* on the return differences: +0.13, +0.59, +0.11; the placebo pair's Sharpe difference (+0.0054, *t* = +0.59) is the same size as the real pairs', which is the whole point.

A 7 bp/yr fee gap inside an arm with 17–22% annualised volatility moves Sharpe in the third decimal. The desk's usual instrument would have found nothing here — and would have been wrong. The right test conditions on the fact that both arms hold *the same index*, which is what differencing the log prices does.

## 10. Live calibration — the estimator is not merely non-zero, it is on the line

Fully **synthetic** and offline: pairs built with a known planted gap, plus noise calibrated to the real tape (transient print error and a slow AR(1) level wobble). The estimator must land on the 45-degree line and stay at zero when nothing was planted.

In [6]:
import numpy as np
from tco import data
panel, truths = data.synthetic_panel(gaps_bp_yr=(0.0, 3.0, 6.0, 12.0), seed=920)
cal = st.panel_calibration(panel, truths, n_boot=300)
print(cal[['planted_bp_yr', 'recovered_bp_yr', 't_annual']].round(2).to_string())
nulls = np.array([
    st.synthetic_detect(data.synthetic_daily(signal_strength=0.0, seed=920 + s)[0],
                        n_boot=200)['t_annual'] for s in range(8)])
print('\nnull x8 seeds: mean t %+.2f (sd %.2f), |t|>=2 on %d/8'
      % (nulls.mean(), nulls.std(ddof=1), int((abs(nulls) >= 2).sum())))

          planted_bp_yr  recovered_bp_yr  t_annual
pair                                              
gap_0bp             0.0            -0.04     -0.02
gap_3bp             3.0             3.41      1.19
gap_6bp             6.0             6.31      3.75
gap_12bp           12.0            13.63      6.62



null x8 seeds: mean t -0.04 (sd 0.34), |t|>=2 on 0/8


## Verdict

- **Signal — Real.** Realised tracking difference of **+5.78 / +6.60 / +7.19 bp/yr** at *t* = **+2.92 / +3.40 / +5.27**, **positive in 5/5 years on each pair** (sign-test *p* = 0.031), Student-*t* intervals clear of zero, within ~2 bp of the published fee gaps, and stable across both halves of QQQM's era cut. The same-fee placebo prints +0.82 bp/yr (*t* = +0.75, 3/5 years) and stays negative for three years in the overlapping race. The synthetic control is calibrated (0 → -0.04, 6 → +6.31, 12 → +13.63) and fires 0/8 on the null — though it plants no mis-timed distributions, so it cannot vouch for the estimator against the defect that actually dominates the real series.
  **Named caveats, all load-bearing:** *(1)* the expensive leg is a **unit investment trust in every pair that shows a gap**, so the measurement is fee + trust cash drag, unseparated; *(2)* the result is **estimator-conditional** (§2b — the cumulative drift puts the placebo at +4.69 above IVV/SPY's +3.38); *(3)* the full pre-2020 histories miss significance because the public adjusted series carries single-year artefacts of -56 bp between same-fee funds (trimming restores +5.48 at *t* = +4.65); *(4)* all four pairs share the **same five years**, so these are not four independent samples; *(5)* the funds are survivors — clones that closed never entered the sample.
- **Tradability — Investable.** Not because the tape pins the number down (it does not) but because the act is a purchase decision with **no forecast, no timing, no turnover and no capacity limit**, on a differential that is contractual and merely confirmed here. Break-even **35–44 trading days** at a 1 bp round-trip differential (42–63 days measured), under nine months at 5 bp — but **74/208/898 days at the honest interval's pessimistic end**. The boundary: worth ~6 bp/yr, so never churn a position for it, and the long/short expression is dead at any borrow above ~5 bp/yr — where a same-fee placebo 'harvests' +3.58 bp/yr, more than two of the three real pairs.
- **Out of scope.** Intraday spreads (not on this tape), options-market depth, tax lots, anything that separates the fee from the trust form, and which wrapper will be cheapest *next* year — fees are a competitive variable, and Study 913 asks the persistence question directly.